# 2. Tool, filesystem e sandbox Docker

Costruiamo un piccolo coding agent realmente funzionante. L'agente può:

- scrivere e leggere file, ma soltanto dentro un workspace temporaneo;
- eseguire comandi in un container Docker effimero;
- osservare exit code, stdout e stderr per correggere il lavoro.

Tutto il codice dei tool è definito nel notebook.

## Configurazione OpenAI

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def find_env() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        candidate = directory / '.env'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('.env non trovato.')


ENV_FILE = find_env()
load_dotenv(ENV_FILE, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(f'OPENAI_API_KEY non configurata in {ENV_FILE}')

MODEL_NAME = os.getenv('OPENAI_MODEL') or 'gpt-5.4-mini'
model = ChatOpenAI(
    model=MODEL_NAME,
    reasoning_effort='low',
    use_responses_api=True,
    store=False,
)
print('Modello:', MODEL_NAME)

## Workspace confinato

`resolve_path` impedisce path traversal: dopo `resolve()`, il percorso deve essere la radice oppure un suo discendente. I tool applicano inoltre limiti di dimensione.

In [ ]:
import tempfile

from langchain.tools import tool

workspace_handle = tempfile.TemporaryDirectory(prefix='langchain-notebook-')
WORKSPACE = Path(workspace_handle.name).resolve()


def resolve_path(relative_path: str) -> Path:
    if not relative_path or Path(relative_path).is_absolute():
        raise ValueError('Usa un percorso relativo non vuoto.')
    candidate = (WORKSPACE / relative_path).resolve()
    if candidate != WORKSPACE and WORKSPACE not in candidate.parents:
        raise ValueError('Percorso fuori dal workspace.')
    return candidate


@tool
def write_text(relative_path: str, content: str) -> str:
    '''Scrive un file UTF-8 nel workspace. Usa percorsi relativi e contenuti sotto 20.000 caratteri.'''
    if len(content) > 20_000:
        raise ValueError('Contenuto troppo grande.')
    target = resolve_path(relative_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
    return f'Scritto {relative_path} ({len(content)} caratteri).'


@tool
def read_text(relative_path: str) -> str:
    '''Legge un file UTF-8 dal workspace e restituisce al massimo 10.000 caratteri.'''
    target = resolve_path(relative_path)
    text = target.read_text(encoding='utf-8')
    return text[:10_000]


print('Workspace temporaneo:', WORKSPACE)

## Tool di esecuzione isolata

Il comando viene interpretato da `sh` dentro il container, non dalla shell host. Il container non ha rete, ha filesystem root in sola lettura, capability rimosse e limiti di memoria, CPU, processi e tempo. L'unico mount scrivibile è il workspace.

In [ ]:
import subprocess


@tool
def docker_exec(command: str) -> str:
    '''Esegue un comando dentro /workspace in un container Python senza rete. Usalo per test e verifiche.'''
    if not command or len(command) > 2_000 or '\x00' in command:
        raise ValueError('Comando non valido.')
    uid = getattr(os, 'getuid', lambda: 10001)()
    gid = getattr(os, 'getgid', lambda: 10001)()
    arguments = [
        'docker', 'run', '--rm',
        '--network', 'none',
        '--read-only',
        '--cap-drop', 'ALL',
        '--security-opt', 'no-new-privileges',
        '--memory', '256m',
        '--cpus', '1',
        '--pids-limit', '64',
        '--user', f'{uid}:{gid}',
        '--env', 'HOME=/tmp',
        '--tmpfs', '/tmp:rw,noexec,nosuid,size=32m',
        '--mount', f'type=bind,src={WORKSPACE},dst=/workspace',
        '--workdir', '/workspace',
        'python:3.12-slim',
        'sh', '-lc', command,
    ]
    try:
        result = subprocess.run(arguments, capture_output=True, text=True, timeout=30, check=False)
    except subprocess.TimeoutExpired:
        return 'ERRORE: timeout dopo 30 secondi.'
    output = f'exit_code={result.returncode}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}'
    return output[:12_000]


print(docker_exec.invoke({'command': 'python --version'}))

## Coding agent completo

Il prompt definisce il risultato e i criteri di successo. L'agente sceglie la sequenza di tool. La verifica è concreta: il programma deve terminare con exit code zero e stampare 385.

In [ ]:
from langchain.agents import create_agent

coding_agent = create_agent(
    model=model,
    tools=[write_text, read_text, docker_exec],
    system_prompt=(
        'Sei un coding agent confinato. Crea file solo con write_text. '
        'Esegui sempre il programma con docker_exec e correggi eventuali errori. '
        'Concludi soltanto dopo exit_code=0.'
    ),
)

result = coding_agent.invoke({
    'messages': [{
        'role': 'user',
        'content': (
            'Crea squares.py: deve calcolare e stampare la somma dei quadrati da 1 a 10. '
            'Eseguilo nel sandbox, verifica che stampi 385 e poi rileggi il sorgente.'
        ),
    }]
})
print(result['messages'][-1].text)

## Verifica indipendente dal modello

In [ ]:
source = (WORKSPACE / 'squares.py').read_text(encoding='utf-8')
verification = docker_exec.invoke({'command': 'python squares.py'})
print(source)
print(verification)

assert 'exit_code=0' in verification
assert '385' in verification
assert any(type(message).__name__ == 'ToolMessage' for message in result['messages'])

## Pulizia ed esperimenti

Il workspace vive finché resta attivo `workspace_handle`. Puoi provare a chiedere `../outside.txt`: il tool deve rifiutarlo. Puoi anche cambiare il programma introducendo intenzionalmente un errore e osservare il ciclo scrittura → esecuzione → correzione.

In [ ]:
workspace_handle.cleanup()
print('Workspace temporaneo eliminato.')